# 12 — Adaptive vs Fixed QEM Statistical Comparison

This notebook evaluates the central research claim:

> Whether a hardware-aware adaptive selection policy can achieve useful reliability improvement with lower or comparable mitigation overhead than fixed strategies.

The comparison is descriptive/statistical, not a political or subjective ranking. The adaptive policy should be evaluated against clearly defined fixed baselines using the same hardware conditions wherever possible.

**No hardware jobs are submitted.**


In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if (ROOT / "Adaptive_QEM_IBM").exists() and not (ROOT / "data").exists():
    ROOT = ROOT / "Adaptive_QEM_IBM"

DATA = ROOT / "data/hardware/extracted"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA:", DATA)


## 1. Load the available experimental results

The notebook first looks for the structured comparison file generated by Notebook 11. If it is unavailable, it stops rather than fabricating a dataset.


In [ ]:
comparison_file = DATA / "qem_comparison_results.csv"

if not comparison_file.exists():
    raise FileNotFoundError(
        "qem_comparison_results.csv is not available. "
        "Complete real IBM hardware execution and Notebook 11 first."
    )

df = pd.read_csv(comparison_file)
display(df.head())
print("Rows:", len(df))


## 2. Required experimental fields

The minimum analysis schema is:

- circuit
- strategy
- success probability
- job ID
- shots

Additional fields such as TVD, distribution fidelity, execution count and overhead should be added when available.


In [ ]:
required = ["circuit", "strategy", "success_probability"]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required fields: {missing}")

df["error_probability"] = 1.0 - df["success_probability"]
display(df)


## 3. Define fixed baselines

The scientifically useful baseline set is:

1. **RAW** — no mitigation.
2. **FIXED-READOUT** — readout mitigation for every benchmark where applicable.
3. **FIXED-ZNE** — ZNE for every benchmark where applicable.
4. **ADAPTIVE** — method selected from circuit and hardware characteristics.

The fixed strategies are comparison baselines, not assumed winners.


In [ ]:
def canonical_strategy(x):
    x = str(x).lower().strip()
    if x in {"raw", "none", "baseline"}:
        return "RAW"
    if "readout" in x:
        return "FIXED-READOUT"
    if "zne" in x:
        return "FIXED-ZNE"
    if "adaptive" in x or "combined" in x:
        return "ADAPTIVE"
    return str(x).upper()

df["comparison_strategy"] = df["strategy"].map(canonical_strategy)

strategy_summary = (
    df.groupby("comparison_strategy")["success_probability"]
      .agg(["count","mean","std","min","max"])
      .reset_index()
)

display(strategy_summary)


## 4. Pairwise improvement against RAW

For each circuit:

\[
\Delta P = P_{strategy} - P_{RAW}
\]

and

\[
	ext{Relative Improvement}
=
rac{P_{strategy}-P_{RAW}}{P_{RAW}}.
\]

Do not interpret an improvement as statistically meaningful without uncertainty/repeated-run evidence.


In [ ]:
raw = (
    df[df["comparison_strategy"] == "RAW"]
    [["circuit","success_probability"]]
    .rename(columns={"success_probability":"raw_success"})
)

paired = df.merge(raw, on="circuit", how="left")
paired["absolute_improvement"] = (
    paired["success_probability"] - paired["raw_success"]
)
paired["relative_improvement"] = (
    paired["absolute_improvement"] /
    paired["raw_success"].replace(0, np.nan)
)

display(paired)


## 5. Aggregate adaptive performance

Compute circuit-level mean, median and dispersion. The adaptive strategy should be judged over the complete benchmark suite rather than selected examples.


In [ ]:
adaptive = paired[paired["comparison_strategy"] == "ADAPTIVE"].copy()

if adaptive.empty:
    print("No adaptive results found yet.")
else:
    adaptive_summary = pd.DataFrame([{
        "n": len(adaptive),
        "mean_success": adaptive["success_probability"].mean(),
        "median_success": adaptive["success_probability"].median(),
        "std_success": adaptive["success_probability"].std(),
        "mean_absolute_improvement": adaptive["absolute_improvement"].mean(),
        "median_absolute_improvement": adaptive["absolute_improvement"].median(),
    }])
    display(adaptive_summary)


## 6. Paired comparison

Because different strategies may be executed on different circuits, compare strategies **within the same benchmark** whenever possible. This avoids confusing circuit difficulty with mitigation performance.


In [ ]:
pivot = (
    paired.pivot_table(
        index="circuit",
        columns="comparison_strategy",
        values="success_probability",
        aggfunc="mean"
    )
)

display(pivot)

if "ADAPTIVE" in pivot.columns and "RAW" in pivot.columns:
    pivot["adaptive_minus_raw"] = pivot["ADAPTIVE"] - pivot["RAW"]

if "ADAPTIVE" in pivot.columns and "FIXED-READOUT" in pivot.columns:
    pivot["adaptive_minus_fixed_readout"] = (
        pivot["ADAPTIVE"] - pivot["FIXED-READOUT"]
    )

if "ADAPTIVE" in pivot.columns and "FIXED-ZNE" in pivot.columns:
    pivot["adaptive_minus_fixed_zne"] = (
        pivot["ADAPTIVE"] - pivot["FIXED-ZNE"]
    )

display(pivot)


## 7. Bootstrap confidence intervals

If the experiment contains one result per circuit, the circuit-level bootstrap gives an empirical uncertainty estimate over the benchmark suite. It is not a substitute for repeated hardware trials.


In [ ]:
def bootstrap_mean(values, n_boot=5000, seed=42):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True)
    means = samples.mean(axis=1)
    return (
        float(np.mean(values)),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )

boot_rows=[]
for strategy, g in paired.groupby("comparison_strategy"):
    m, lo, hi = bootstrap_mean(g["success_probability"])
    boot_rows.append({
        "strategy": strategy,
        "mean_success": m,
        "bootstrap_95_low": lo,
        "bootstrap_95_high": hi,
        "n": len(g),
    })

bootstrap_df = pd.DataFrame(boot_rows)
display(bootstrap_df)


## 8. McNemar test for deterministic success, when repeated paired binary outcomes exist

For individual-shot binary outcomes, McNemar's test can compare paired methods. This notebook does not invent shot-level paired outcomes from aggregate counts. Therefore, this section only activates if a future dataset provides paired binary observations.


In [ ]:
paired_binary_file = DATA / "paired_binary_outcomes.csv"

if not paired_binary_file.exists():
    print(
        "No shot-level paired binary dataset found. "
        "McNemar analysis is intentionally skipped."
    )
else:
    pb = pd.read_csv(paired_binary_file)
    print("Loaded paired binary outcomes:", pb.shape)
    print("Expected columns: circuit, raw_success, adaptive_success")


## 9. Overhead analysis

Reliability cannot be reported without experimental cost.

For each strategy record:

\[
O = rac{N_{physical\ executions}}{N_{baseline\ executions}}.
\]

For example, three ZNE scale factors imply at least three physical circuit executions per benchmark before accounting for additional calibration overhead.


In [ ]:
overhead_file = ROOT / "data/hardware/adaptive_qem_execution_matrix.csv"

if overhead_file.exists():
    om = pd.read_csv(overhead_file)

    overhead = (
        om.groupby(["circuit","execution_method"])
          .size()
          .reset_index(name="execution_rows")
    )

    display(overhead)

    overhead.to_csv(
        RESULTS / "qem_execution_overhead_by_circuit.csv",
        index=False
    )
else:
    print("Adaptive execution matrix not found.")


## 10. Adaptive policy coverage

Measure how often the selector chooses each method. This describes the operating regime of the policy and is not itself evidence that a method is superior.


In [ ]:
selection_file = ROOT / "data/hardware/adaptive_qem_selection_plan.csv"

if selection_file.exists():
    sel = pd.read_csv(selection_file)

    coverage = (
        sel["method"]
        .value_counts(dropna=False)
        .rename_axis("selected_method")
        .reset_index(name="circuits")
    )

    coverage["fraction"] = coverage["circuits"] / coverage["circuits"].sum()
    display(coverage)

    coverage.to_csv(
        RESULTS / "adaptive_policy_coverage.csv",
        index=False
    )
else:
    print("Selection plan not found.")


## 11. Publication-ready summary

This table is intentionally descriptive. It reports measured quantities without declaring an overall winner.


In [ ]:
summary = (
    paired.groupby("comparison_strategy")
    .agg(
        circuits=("circuit","nunique"),
        mean_success=("success_probability","mean"),
        median_success=("success_probability","median"),
        mean_error=("error_probability","mean"),
        mean_improvement=("absolute_improvement","mean"),
    )
    .reset_index()
)

summary.to_csv(RESULTS / "adaptive_vs_fixed_qem_summary.csv", index=False)
display(summary)


## 12. Interpretation framework for the paper

Use the following structure in the manuscript:

- **Reliability:** report success probability/fidelity/TVD changes.
- **Adaptivity:** report the hardware/circuit conditions associated with each selection.
- **Overhead:** report physical executions, shots, calibration cost and runtime where available.
- **Uncertainty:** report confidence intervals and repeated-run variability.
- **Limitations:** state benchmark size, backend dependence, calibration drift and rule-based threshold assumptions.
- **Validation:** do not claim the selector is universally optimal; establish only what the measured experimental dataset supports.
